# Step 2: generate BLER tables on GPU

Before running: **Settings > Accelerator > GPU** (T4 or P100) and **Internet on**. Add your uploaded `mlla` dataset as input.

In [ ]:
import os, shutil
src = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'make_tables.py' in files:
        src = os.path.dirname(root)
        break
assert src, "Project not found in /kaggle/input: attach the dataset (latest version) and restart the session"
shutil.copytree(src, '/kaggle/working/mlla', dirs_exist_ok=True)
os.chdir('/kaggle/working/mlla')
print("working in", os.getcwd())

In [ ]:
!pip install -q sionna-no-rt
import torch
print(torch.__version__, 'CUDA available:', torch.cuda.is_available())
# Must print True. If False, check the accelerator setting and restart.

In [ ]:
# 1) quick check that the chain runs on GPU (prints device=cuda)
!python -m scripts.smoke_test

In [ ]:
# 2) all 52 scenarios (channels x impairments); long, use Save Version > Save & Run All. Watch the per-scenario time it prints.
!python -m scripts.make_tables --preset full --n-tb 256 --batch 256

In [ ]:
# 3) plots + OLLA check on the real tables
!python -m scripts.plot_tables results/tables/bler_fits.json
!python -m scripts.run_olla --snr 10 20

In [ ]:
# 4) package results to download (Output tab, right panel)
!cd results && zip -qr /kaggle/working/tables.zip tables
print('download /kaggle/working/tables.zip')

Later, for the impairment scenarios: `!python -m scripts.make_tables --preset full`. It skips scenarios that already exist, so it is safe to rerun if the session times out.